In [ ]:
# - .env의 postgresql 접속 정보 불러오기
# - 실제 db 연결이 가능한지 select 1로 확인

import os
import pandas as pd

from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL as url


# 현재 실행 위치를 기준으로 프로젝트 루트 경로 설정
current_dir = os.getcwd()

if os.path.basename(current_dir).lower() == 'notebooks':
	project_root = os.path.dirname(current_dir)
else:
	project_root = current_dir

load_dotenv(
	os.path.join(project_root, '.env'),
	override=True
)


# 비밀번호에 특수문자가 있어도 안전하게 접속 url 생성
db_url = url.create(
	drivername='postgresql+psycopg2',
	username=os.getenv('db_user'),
	password=os.getenv('db_password'),
	host=os.getenv('db_host'),
	port=int(os.getenv('db_port')),
	database=os.getenv('db_name')
)

engine = create_engine(db_url)


# db 연결 확인
with engine.connect() as conn:
	result = conn.execute(text('select 1'))
	print(result.scalar())

1


In [ ]:
# - 분석에서 사용할 두 mart의 schema 위치 확인

query = """
select
	table_schema,
	table_name
from information_schema.views
where table_name in (
	'customer_behavior_mart',
	'order_level_mart'
)
order by table_name;
"""

mart_location = pd.read_sql(query, engine)

mart_location

,table_schema,table_name
0,public,customer_behavior_mart
1,public,order_level_mart


In [ ]:
# - 현재 연결된 database를 확인

query = """
select
	current_database() as database_name,
	n.nspname as table_schema,
	c.relname as table_name,
	case
		when c.relkind = 'r' then 'table'
		when c.relkind = 'v' then 'view'
		when c.relkind = 'm' then 'materialized_view'
		else c.relkind::text
	end as object_type
from pg_class c
	inner join pg_namespace n
		on c.relnamespace = n.oid
where c.relname in (
	'customer_behavior_mart',
	'order_level_mart'
)
order by c.relname;
"""

mart_location = pd.read_sql(query, engine)

mart_location

,database_name,table_schema,table_name,object_type
0,olist,public,customer_behavior_mart,view
1,olist,public,order_level_mart,view


In [ ]:
# - 두 mart의 실제 컬럼명을 확인

query = """
select
	table_name,
	ordinal_position,
	column_name,
	data_type
from information_schema.columns
where table_schema = 'public'
	and table_name in (
		'customer_behavior_mart',
		'order_level_mart'
	)
order by
	table_name,
	ordinal_position;
"""

mart_columns = pd.read_sql(query, engine)

mart_columns

,table_name,ordinal_position,column_name,data_type
0,customer_behavior_mart,1,customer_unique_id,character varying
1,customer_behavior_mart,2,first_order_id,text
2,customer_behavior_mart,3,first_order_timestamp,text
3,customer_behavior_mart,4,second_order_timestamp,text
4,customer_behavior_mart,5,last_order_timestamp,text
5,customer_behavior_mart,6,order_count,bigint
6,customer_behavior_mart,7,total_payment,real
7,customer_behavior_mart,8,average_order_value,double precision
8,customer_behavior_mart,9,first_order_payment_total,real
9,customer_behavior_mart,10,days_to_second_order,integer


In [ ]:
# - retention validation에 사용할 고객 단위 핵심 컬럼 불러오기
# - 고객 수와 데이터 형태가 sql mart와 일치하는지 확인

query = """
select
	customer_unique_id,
	first_order_id,
	first_order_payment_total,
	eligible_90d,
	repurchase_90d,
	first_order_review_score,
	first_order_delayed_flag,
	cohort_month
from customer_behavior_mart;
"""

retention_base = pd.read_sql(query, engine)

print(retention_base.shape)
retention_base.head()

(93358, 8)


,customer_unique_id,first_order_id,first_order_payment_total,eligible_90d,repurchase_90d,first_order_review_score,first_order_delayed_flag,cohort_month
0,0000f46a3911fa3c0805444483337064,b33ec3b699337181488304f362a6b734,86.22,1,0.0,3.0,0.0,2017-03-01
1,0000f6ccb0745a6a4b88665a16c9f078,41272756ecddd9a9ed0180413cc22fb6,43.62,1,0.0,4.0,0.0,2017-10-01
2,00050ab1314c0e55a6ca13cf7181fecf,d0028facea13f508e880202d7097a5a1,35.38,1,0.0,4.0,0.0,2018-04-01
3,00053a61a98854899e70ed204dd4bafe,44e608f2db00c74a1fe329de44416a4e,419.18,1,0.0,1.0,0.0,2018-02-01
4,0005e1862207bf6ccc02e4228effd9a0,ae76bef74b97bcb0b3e355e60d9a6f9c,150.12,1,0.0,4.0,0.0,2017-03-01
